# Home Repair Trend Pipeline — v0.2
**Fixed:** Replaced pytrends with direct HTTP calls to Google Trends (avoids urllib3 compatibility bug and gives us full header control)

**Sources:** YouTube Data API v3 · Google Trends (direct requests)  

---
### If you just ran the diagnostic, wait 5-10 minutes before running the Trends cells.
Google issued a 429 rate limit. It clears quickly.

In [ ]:
# Cell 1 — Install dependencies (pytrends no longer needed)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'google-api-python-client',
    'pandas',
    'requests',
    'python-dotenv'
])
print('Dependencies installed.')

In [ ]:
# Cell 2 — Configuration
import os
from dotenv import load_dotenv

load_dotenv()

YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY', 'PASTE_YOUR_KEY_HERE')

if YOUTUBE_API_KEY == 'PASTE_YOUR_KEY_HERE':
    print('⚠️  No YouTube API key found.')
else:
    print(f'✅ YouTube API key loaded: {YOUTUBE_API_KEY[:12]}...')

WEIGHT_YOUTUBE = 0.55
WEIGHT_TRENDS  = 0.45

YT_RESULTS_PER_TOPIC = 10

# Seconds to sleep between Google Trends requests.
# Google allows roughly 1 request per 5-10 seconds from the same IP.
# If you get 429s, increase this.
TRENDS_SLEEP = 8

In [ ]:
# Cell 3 — Seed topics
SEED_TOPICS = [
    'garage door not closing',
    'water heater making noise',
    'bathroom exhaust fan replacement',
    'drywall hole repair',
    'leaking shower grout fix',
    'toilet running constantly',
    'exterior door weatherstripping',
    'HVAC filter replacement',
    'circuit breaker tripping',
    'roof shingle damage assessment',
]

print(f'Scoring {len(SEED_TOPICS)} topics.')

In [ ]:
# Cell 4 — YouTube functions (unchanged, already working)
from googleapiclient.discovery import build
from datetime import datetime, timedelta, timezone
import time
import pandas as pd

def build_youtube_client(api_key):
    return build('youtube', 'v3', developerKey=api_key)

def fetch_youtube_videos(client, query, max_results=10):
    try:
        search_resp = client.search().list(
            q=query,
            part='id,snippet',
            type='video',
            maxResults=max_results,
            order='viewCount',
            relevanceLanguage='en',
        ).execute()

        video_ids = [
            item['id']['videoId']
            for item in search_resp.get('items', [])
            if item['id'].get('videoId')
        ]
        if not video_ids:
            return []

        stats_resp = client.videos().list(
            id=','.join(video_ids),
            part='statistics,snippet',
        ).execute()

        videos = []
        for item in stats_resp.get('items', []):
            published_raw = item['snippet'].get('publishedAt', '')
            try:
                published_at = datetime.fromisoformat(published_raw.replace('Z', '+00:00'))
            except ValueError:
                published_at = None

            videos.append({
                'video_id':    item['id'],
                'title':       item['snippet']['title'],
                'published_at': published_at,
                'view_count':  int(item['statistics'].get('viewCount', 0)),
                'channel':     item['snippet'].get('channelTitle', ''),
            })
        return videos

    except Exception as e:
        print(f'  YouTube error: {e}')
        return []

def compute_youtube_velocity(videos, recent_days=90):
    if not videos:
        return 0.0
    now = datetime.now(timezone.utc)
    cutoff = now - timedelta(days=recent_days)
    recent, older = [], []
    for v in videos:
        if v['published_at'] is None:
            continue
        age_days = max((now - v['published_at']).days, 1)
        vpd = v['view_count'] / age_days
        (recent if v['published_at'] >= cutoff else older).append(vpd)

    avg_recent = sum(recent) / len(recent) if recent else 0
    avg_older  = sum(older)  / len(older)  if older  else 0
    ratio = avg_recent / (avg_older + 1)
    return round(min(ratio / (ratio + 1), 1.0), 4)

print('YouTube functions ready.')

In [ ]:
# Cell 5 — Google Trends: direct HTTP (replaces pytrends entirely)
#
# How this works:
#   Google Trends has an internal JSON API. We first hit the explore
#   endpoint to get a one-time token, then use that token to fetch
#   the actual interest-over-time data. We use a real browser
#   User-Agent so Google doesn't block us as a bot.
#
#   This is the same thing pytrends does internally, but without
#   the broken urllib3 retry logic.

import requests
import json
import re
import numpy as np

TRENDS_SESSION = requests.Session()
TRENDS_SESSION.headers.update({
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Referer': 'https://trends.google.com/',
})

# Warm up the session — visit the homepage to get cookies.
# This is important: Google checks for the NID/CONSENT cookie.
print('Warming up Google Trends session...')
_warmup = TRENDS_SESSION.get('https://trends.google.com/', timeout=10)
print(f'Warm-up status: {_warmup.status_code}')
time.sleep(2)


def fetch_trends_data(keyword, timeframe='today 3-m', geo='US'):
    """
    Returns a list of (date_str, value) tuples, or [] on failure.
    
    Step 1: POST to /trends/api/explore to get a one-time widget token.
    Step 2: GET /trends/api/widgetdata/multiline with that token.
    """
    explore_url = 'https://trends.google.com/trends/api/explore'
    payload = {
        'hl': 'en-US',
        'tz': '360',
        'req': json.dumps({
            'comparisonItem': [{
                'keyword': keyword,
                'geo': geo,
                'time': timeframe,
            }],
            'category': 0,
            'property': '',
        }),
    }

    try:
        resp = TRENDS_SESSION.get(explore_url, params=payload, timeout=15)

        if resp.status_code == 429:
            print(f'  429 rate limit — waiting 30s...')
            time.sleep(30)
            resp = TRENDS_SESSION.get(explore_url, params=payload, timeout=15)

        if resp.status_code != 200:
            print(f'  Trends explore failed: HTTP {resp.status_code}')
            return []

        # Response starts with ")]}'\n" — strip it
        raw = resp.text
        clean = raw[raw.index('\n')+1:] if '\n' in raw else raw
        data = json.loads(clean)

        # Find the TIMESERIES widget token
        widgets = data.get('widgets', [])
        token = None
        req_obj = None
        for w in widgets:
            if w.get('id') == 'TIMESERIES':
                token = w.get('token')
                req_obj = w.get('request')
                break

        if not token:
            print(f'  No TIMESERIES widget found in response')
            return []

    except Exception as e:
        print(f'  Trends explore error: {e}')
        return []

    # Step 2: fetch the actual time series data
    time.sleep(1)
    multiline_url = 'https://trends.google.com/trends/api/widgetdata/multiline'
    multiline_params = {
        'hl': 'en-US',
        'tz': '360',
        'req': json.dumps(req_obj),
        'token': token,
        'user_type': 'USER_TYPE_SCRAPER',
    }

    try:
        resp2 = TRENDS_SESSION.get(multiline_url, params=multiline_params, timeout=15)

        if resp2.status_code == 429:
            print(f'  429 on multiline — waiting 30s...')
            time.sleep(30)
            resp2 = TRENDS_SESSION.get(multiline_url, params=multiline_params, timeout=15)

        if resp2.status_code != 200:
            print(f'  Trends multiline failed: HTTP {resp2.status_code}')
            return []

        raw2 = resp2.text
        clean2 = raw2[raw2.index('\n')+1:] if '\n' in raw2 else raw2
        data2 = json.loads(clean2)

        points = []
        for row in data2.get('default', {}).get('timelineData', []):
            date_str = row.get('formattedAxisTime', row.get('formattedTime', ''))
            value = row.get('value', [0])[0]
            points.append((date_str, value))

        return points

    except Exception as e:
        print(f'  Trends multiline error: {e}')
        return []


def compute_trends_growth(keyword):
    """
    Fetch 90 days, compare last third vs first two thirds.
    Returns normalized 0-1 score.
    """
    points = fetch_trends_data(keyword, timeframe='today 3-m', geo='US')

    if len(points) < 8:
        print(f'  Not enough data points: {len(points)}')
        return 0.0

    values = np.array([p[1] for p in points], dtype=float)
    split = len(values) * 2 // 3
    baseline_avg = np.mean(values[:split]) + 0.01
    recent_avg   = np.mean(values[split:])
    ratio = recent_avg / baseline_avg
    return round(min(ratio / (ratio + 1), 1.0), 4)


print('Trends functions ready (no pytrends, no urllib3 issues).')

# Quick smoke test
print('\nSmoke test: fetching "garage door" trend...')
test_pts = fetch_trends_data('garage door', timeframe='today 3-m', geo='US')
if test_pts:
    print(f'✅ Got {len(test_pts)} data points. Last value: {test_pts[-1]}')
else:
    print('❌ Still getting empty data. See note below.')
    print('   Wait 10 minutes and re-run — the 429 from earlier is still clearing.')

In [ ]:
# Cell 6 — Run the full pipeline
results = []
yt_client = build_youtube_client(YOUTUBE_API_KEY)

for i, topic in enumerate(SEED_TOPICS):
    print(f'[{i+1}/{len(SEED_TOPICS)}] "{topic}"')

    # YouTube
    yt_videos   = fetch_youtube_videos(yt_client, topic, max_results=YT_RESULTS_PER_TOPIC)
    yt_velocity = compute_youtube_velocity(yt_videos)
    print(f'       YT velocity:    {yt_velocity:.3f}  ({len(yt_videos)} videos)')

    # Trends — sleep between calls to stay under rate limit
    time.sleep(TRENDS_SLEEP)
    trends_growth = compute_trends_growth(topic)
    print(f'       Trends growth:  {trends_growth:.3f}')

    score = round(yt_velocity * WEIGHT_YOUTUBE + trends_growth * WEIGHT_TRENDS, 4)
    print(f'       Score:          {score:.3f}\n')

    results.append({
        'topic':            topic,
        'youtube_velocity': yt_velocity,
        'trends_growth':    trends_growth,
        'score':            score,
        'yt_videos':        len(yt_videos),
    })

print('Pipeline complete.')

In [ ]:
# Cell 7 — Results
import json

df = pd.DataFrame(results).sort_values('score', ascending=False).reset_index(drop=True)
print('=== TREND SCORES ===')
print(df[['topic','youtube_velocity','trends_growth','score']].to_string())

output = df[['topic','youtube_velocity','trends_growth','score']].to_dict(orient='records')
with open('trend_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('\nSaved: trend_results.json')
print(json.dumps(output[:3], indent=2))